# Homework 4

## Consegna

 ### 1. Compressione di immagini

Caricare e visualizzare un’immagine (diversa
dal cameraman) in scala di grigio con matrice associata A di dimensione $mxn$. Fissati alcuni valori di $p$:


Inclusione librerie

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from skimage.color import rgb2gray

# Importazione moduli
try:
    from ProblemiInversi import operators, solvers, utilities
except ImportError:
    import operators, solvers, utilities

# =============================================================================
# CARICAMENTO IMMAGINE (COMUNE A TUTTO)
# =============================================================================
filename = 'hulk.bmp'  # Cambia con il tuo file

try:
    image_source = io.imread(filename)
except FileNotFoundError:
    print(f"ERRORE: Non trovo il file '{filename}'.")
    raise

# Conversione in scala di grigi
if image_source.ndim == 3:
    A_img = rgb2gray(image_source) # Chiamiamola A_img per evitare confusions
else:
    A_img = image_source

# Normalizzazione
if A_img.max() > 1.0:
    A_img = A_img / 255.0

m, n = A_img.shape

plt.figure(figsize=(6, 6))
plt.imshow(A_img, cmap='gray')
plt.title(f"Immagine Originale ({m}x{n})")
plt.axis('off')
plt.show()


# =============================================================================
# PARTE 1: COMPRESSIONE SVD
# =============================================================================

def Ap(U, s, Vt, p):
    S_p = np.diag(s[:p])
    A_p = U[:, :p] @ S_p @ Vt[:p, :]
    return A_p

def cp(shape, p):
    # shape[0] = m, shape[1] = n
    return (1/p) * min(shape) - 1

def plot_svd_results(p_values, errors, compression_factors):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    color = 'tab:red'
    ax1.set_xlabel('Valore di p (rango)')
    ax1.set_ylabel('Errore Relativo', color=color)
    ax1.plot(p_values, errors, marker='o', color=color, label='Errore Relativo')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True)

    ax2 = ax1.twinx()  # Secondo asse y per il fattore di compressione
    color = 'tab:blue'
    ax2.set_ylabel('Fattore di Compressione (cp)', color=color)
    ax2.plot(p_values, compression_factors, marker='s', linestyle='--', color=color, label='Fattore Compressione')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title("SVD: Errore Relativo e Fattore di Compressione")
    plt.show()

def svd_compression(A, p_values):
    print("\n--- INIZIO PARTE 1: SVD ---")
    U, s, Vt = np.linalg.svd(A, full_matrices=False)

    errors = []
    compression_factors = []

    plt.figure(figsize=(15, 8))

    valid_p = [p for p in p_values if p <= min(A.shape)]
    
    for i, p in enumerate(valid_p):
        # Calcolo Ap
        A_p = Ap(U, s, Vt, p)
        
        # Errori e Compressione
        norm_A = np.linalg.norm(A, 'fro')
        rel_err = np.linalg.norm(A - A_p, 'fro') / norm_A
        errors.append(rel_err)
        
        comp_fact = cp(A.shape, p)
        compression_factors.append(comp_fact)
        
        # Visualizzazione
        plt.subplot(2, 3, i+1)
        plt.imshow(A_p, cmap='gray')
        plt.title(f"p={p}\nErr: {rel_err:.4f}, Cp: {comp_fact:.2f}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()
    
    plot_svd_results(valid_p, errors, compression_factors)

# Esecuzione Parte 1
p_values = [5, 20, 50, 100, 150] 
svd_compression(A_img, p_values)


# =============================================================================
# PARTE 2: IMAGE DEBLUR
# =============================================================================

def prob_test(nl, sigma, x_true):
    # 1. Creazione Operatore di Blur
    k_size = 15
    kernel = utilities.gaussian2d_kernel(k_size, sigma)
    A = operators.ConvolutionOperator(kernel)
    
    # Calcolo immagine sfocata pulita
    b_clean = A(x_true)
    
    # 2. Aggiunta Rumore (CORRETTO: Somma immagine + rumore)
    noise = utilities.gaussian_noise(b_clean, nl)
    b_noisy = b_clean + noise
    
    # Calcolo norma del rumore (Delta)
    delta = np.linalg.norm(noise)
    
    print(f"\n--- Livello di Rumore: {nl*100}% (Delta: {delta:.4f}) ---")
    return b_noisy, A, delta

def CGLS(b_noisy, x_true, A, max_iter_cgls):
    print("Calcolo soluzione Naive (CGLS)...")
    solver_naive = solvers.CGLS(A) 
    x0 = np.zeros_like(b_noisy)
    x_naive = solver_naive.solve(b_noisy, x0, kmax=max_iter_cgls)
        
    err_naive = utilities.rel_err(x_naive, x_true)
    print(f"-> Naive Error: {err_naive:.4f}")
    return x_naive, err_naive

def Tik(b_noisy, x_true, A, max_iter_cgls, lambdas_tik, delta, x0):
    print("Calcolo soluzione Tikhonov...")
        
    best_err_tik = float('inf')
    best_lam_tik = 0
    best_x_tik = None
        
    tik_errors = []
    tik_residuals = [] 
    x_tik_disc = None
    best_disc_diff = float('inf')
    lam_disc_tik = 0
        
    L = operators.Identity() 
       
    for lam in lambdas_tik:
        solver_tik = solvers.CGLS(A, L=L, lmbda=lam)
        x_tik = solver_tik.solve(b_noisy, x0, kmax=max_iter_cgls)
          
        # Errore vs GT
        err = utilities.rel_err(x_tik, x_true)
        tik_errors.append(err)
           
        if err < best_err_tik:
            best_err_tik = err
            best_lam_tik = lam
            best_x_tik = x_tik
            
        # Residuo per Discrepanza ||Ax - y||
        res_norm = np.linalg.norm(A(x_tik) - b_noisy)
        tik_residuals.append(res_norm)
            
        diff = abs(res_norm - delta)
        if diff < best_disc_diff:
            best_disc_diff = diff
            x_tik_disc = x_tik
            lam_disc_tik = lam

    print(f"-> Tikhonov Best (vs GT): Lambda={best_lam_tik:.5f}, Err={best_err_tik:.4f}")
    if x_tik_disc is not None:
        print(f"-> Tikhonov Discrepancy: Lambda={lam_disc_tik:.5f}, Err={utilities.rel_err(x_tik_disc, x_true):.4f}")
    
    return best_x_tik, tik_errors, x_tik_disc, best_err_tik

def TV(b_noisy, x_true, A, lambdas_tv, x0):
    print("Calcolo soluzione Total Variation (Gradient Descent)...")
    
    # 1. Configurazione Solver
    beta = 1e-3
    gd_tv_solver = solvers.GDTotalVariation(A, beta=beta)

    # 2. Parametri Iterativi Fissi
    kmax = 30     # Iterazioni ridotte
    tolf = 1e-8
    tolx = 1e-8
    
    best_err_tv = float('inf')
    best_lam_tv = 0
    best_x_tv = None
    tv_errors = []
        
    for lam in lambdas_tv:
        # x0 passato per velocizzare
        x_tv, _, _ = gd_tv_solver.solve(b_noisy, lam, x0, kmax, tolf, tolx)
            
        err = utilities.rel_err(x_tv, x_true)
        tv_errors.append(err)
            
        if err < best_err_tv:
            best_err_tv = err
            best_lam_tv = lam
            best_x_tv = x_tv
            
    print(f"-> TV Best (vs GT): Lambda={best_lam_tv:.5f}, Err={best_err_tv:.4f}")
    
    return best_x_tv, tv_errors, best_err_tv

def plot_deblur_results(b_noisy, x_naive, err_naive, 
                 best_x_tik, best_err_tik, x_tik_disc, 
                 best_x_tv, best_err_tv, 
                 lambdas_tik, tik_errors, lambdas_tv, tv_errors, nl, sigma, x_true):
    
    plt.figure(figsize=(15, 5))
    
    titles = ["Dati Rumorosi", "Naive", "Tikhonov (Best)", "Tikhonov (Disc)", "TV (Best)"]
    images = [b_noisy, x_naive, best_x_tik, x_tik_disc, best_x_tv]
    
    # Calcolo errore discrepanza vs Ground Truth (se esiste la soluzione)
    if x_tik_disc is not None:
        err_disc = utilities.rel_err(x_tik_disc, x_true) 
    else:
        err_disc = 0
        
    errors = [None, err_naive, best_err_tik, err_disc, best_err_tv]

    for i in range(5):
        plt.subplot(1, 5, i+1)
        if images[i] is not None:
            plt.imshow(images[i], cmap='gray')
            t = titles[i]
            if errors[i] is not None:
                t += f"\nErr: {errors[i]:.3f}"
            plt.title(t)
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Plot Curve Errori
    plt.figure(figsize=(8, 4))
    plt.semilogx(lambdas_tik, tik_errors, 'b-o', label='Tikhonov')
    plt.semilogx(lambdas_tv, tv_errors, 'r-s', label='TV')
    plt.axhline(y=err_naive, color='k', linestyle='--', label='Naive Base')
    plt.xlabel('Lambda')
    plt.ylabel('Relative Error')
    plt.title(f'Errore vs Lambda (Sigma={sigma}, Noise={nl})')
    plt.legend()
    plt.grid(True)
    plt.show()

def test(x_true, max_iter_cgls, lambdas_tik, lambdas_tv, sigmas, noise_levels):
    print("\n--- INIZIO PARTE 2: DEBLUR ---")
    
    for sigma in sigmas:
        print(f"\n{'='*80}")
        print(f"ANALISI CON PSF SIGMA = {sigma}")
        print(f"{'='*80}")
        
        for nl in noise_levels:
            # 1. Genera problema
            b_noisy, A_op, delta = prob_test(nl, sigma, x_true) 
            
            # Punto di partenza ottimizzato (b_noisy invece di zero)
            x0_start = b_noisy.copy()
            
            # 2. Esegui Solutori
            x_naive, err_naive = CGLS(b_noisy, x_true, A_op, max_iter_cgls)
            
            best_x_tik, tik_errors, x_tik_disc, best_err_tik = Tik(
                b_noisy, x_true, A_op, max_iter_cgls, lambdas_tik, delta, x0_start
            )
            
            best_x_tv, tv_errors, best_err_tv = TV(
                b_noisy, x_true, A_op, lambdas_tv, x0_start
            )
            
            # 3. Plot (Passando x_true esplicitamente!)
            plot_deblur_results(b_noisy, x_naive, err_naive, 
                         best_x_tik, best_err_tik, x_tik_disc, 
                         best_x_tv, best_err_tv, 
                         lambdas_tik, tik_errors, lambdas_tv, tv_errors, nl, sigma, x_true)


# Parametri globali
max_iter_cgls = 100
lambdas_tik = np.logspace(-4, 0, 10)
lambdas_tv = np.logspace(-4, 0, 10)
sigmas = [1, 3]      # Esempio ridotto
noise_levels = [0.05] # Esempio ridotto

# Esecuzione Parte 2
test(A_img, max_iter_cgls, lambdas_tik, lambdas_tv, sigmas, noise_levels)